# 10 — Subject-Independent (LOSO) Benchmark

**One driver for the whole model zoo under the leave-one-subject-out protocol**, so
every algorithm is scored identically and a comparative analysis can be produced in
one place.

Each of the 23 folds trains a fresh model on 22 subjects and tests on the held-out
23rd; normalisation statistics are fit on the training subjects only. We report the
**mean ± std of the headline metrics across the 23 folds**. A `MajorityBaseline`
(predict the training-majority class) is scored on the same folds as a reference
floor — on DREAMER the subject-independent numbers sit close to it, which is the
whole point of the comparison.

All logic lives in [`src/benchmark_si.py`](../src/benchmark_si.py):

* `REGISTRY` — the per-model LOSO recipe (feature view, batch, epochs, lr, custom
  training loop). Single source of truth: the model notebooks 01–08 call
  `run_one`, which reads the same table.
* `run_one(name, ...)` — one model, 23-fold LOSO, writes
  `results/{name}_results.json` (`loso` section).
* `run_all(...)` — every registered model, then rebuilds the comparison table and
  figures.
* `build_comparison(...)` / `make_plots(...)` — the comparative analysis.

The same run is available from the shell:

```bash
python -m src.benchmark_si --quick                 # 3-fold / 5-epoch smoke test
python -m src.benchmark_si                          # full 23-fold run, every model
python -m src.benchmark_si --models EEGNet DGCNN    # subset
python -m src.benchmark_si --label arousal          # switch target
python -m src.benchmark_si --aggregate-only         # rebuild table + plots only
```


## 1. Setup

In [ ]:
import os, sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import pandas as pd
from IPython.display import Image, display

from src import benchmark_si as B
from src.evaluation import get_device

pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print("device       :", get_device())
print("registry     :", list(B.REGISTRY))
print("results dir  :", B.RESULTS_DIR)

# per-model LOSO recipe (single source of truth)
pd.DataFrame([
    {"model": s.name, "family": s.family, "feature": s.feature,
     "channels": "5-ch" if s.channels else "14-ch", "batch": s.batch_size,
     "epochs": s.epochs, "lr": s.lr, "val_frac": s.val_frac, "runner": s.runner}
    for s in B.REGISTRY.values()
])

## 2. Configuration

* `QUICK = True` → 3 folds × 5 epochs (≈ a few minutes) to validate the pipeline.
* `QUICK = False` → the full 23-fold benchmark. This is a **multi-hour** run on a
  laptop GPU/CPU; launch it from the shell with `nohup` / `tmux` if you can.
* `MODELS = None` runs every registered model; set a list to restrict.
* `SKIP_EXISTING = True` skips a model whose `results/*.json` already matches
  `(LABEL, QUICK)` — so you can resume an interrupted run.

In [ ]:
QUICK         = True
LABEL         = "valence"          # "valence" | "arousal" | "dominance"
MODELS        = None               # e.g. ["EEGNet", "DGCNN"]  (None = all)
SKIP_EXISTING = True
WITH_BASELINE = True

## 3. Run the protocol

In [ ]:
results = B.run_all(
    models=MODELS, label=LABEL, quick=QUICK,
    skip_existing=SKIP_EXISTING, with_baseline=WITH_BASELINE,
)
print("\nran this call:", list(results))

## 4. Comparative analysis

`build_comparison` reads **every** `results/*_results.json` with a `loso` section
(so models scored in earlier runs or from their own notebooks are included), ranks
by LOSO accuracy, and writes `results/subject_independent_comparison.{csv,md}`.

In [ ]:
df = B.build_comparison(B.RESULTS_DIR, label=LABEL)
df[["model", "family", "params", "acc", "acc_std", "f1", "f1_std",
    "precision", "recall", "min_fold_acc", "max_fold_acc", "n_folds",
    "train_time_sec"]]

### 4.1 Figures

In [ ]:
B.make_plots(B.RESULTS_DIR, label=LABEL)
for fn in ["si_accuracy_by_model.png", "si_per_subject_heatmap.png",
           "si_accuracy_vs_params.png"]:
    p = os.path.join(B.RESULTS_DIR, "figures", fn)
    if os.path.exists(p):
        display(Image(filename=p))

## 5. Notes for the methodology chapter

* Every metric comes from `src/evaluation.compute_metrics` (accuracy, macro
  precision / recall / F1, confusion matrix) — no per-notebook metric code.
* The 23-fold **std is the honest uncertainty band** for a subject-independent
  claim on DREAMER; report `mean ± std`, not a single number.
* `MajorityBaseline` and the `chance (0.50)` line are the two references every
  model must clear to support a "learns subject-transferable structure" claim.
* `results/{Model}_results.json` also logs `params_count` and `train_time_sec`
  for the accuracy-vs-efficiency analysis (figure `si_accuracy_vs_params.png`).
* To reproduce with a different target, set `LABEL` and re-run, or
  `python -m src.benchmark_si --label arousal`.
